# Model 3: DermNet Training

This notebook trains a ResNet152V2 model on the DermNet dataset.
**Optimized for Google Colab T4 GPU**

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_SAVE_DIR = '/content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install kagglehub -q

import kagglehub
import numpy as np
import json
import shutil
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet152V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# Download DermNet
print("Downloading DermNet...")
DERMNET_PATH = kagglehub.dataset_download("shubhamgoel27/dermnet")
print(f"Dataset Path: {DERMNET_PATH}")

100%|██████████| 1.72G/1.72G [01:19<00:00, 23.1MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/shubhamgoel27/dermnet/versions/1


In [4]:
# =============================================================================
# DEFINE THE 7 CLASSES TO KEEP (Add this BEFORE the train/test split section)
# =============================================================================

# Only train on these 7 non-overlapping classes
CLASSES_TO_KEEP = [
    'Acne and Rosacea Photos',
    'Nail Fungus and other Nail Disease',
    'Light Diseases and Disorders of Pigmentation',
    'Vascular Tumors',
    'Cellulitis Impetigo and other Bacterial Infections',
    'Scabies Lyme Disease and other Infestations and Bites',
    'Lupus and other Connective Tissue diseases'
]

print("="*70)
print("DermNet Training Configuration")
print("="*70)
print(f"\nTraining on {len(CLASSES_TO_KEEP)} selected classes:")
for i, cls in enumerate(CLASSES_TO_KEEP, 1):
    print(f"  {i}. {cls}")
print("\n" + "="*70 + "\n")

DermNet Training Configuration

Training on 7 selected classes:
  1. Acne and Rosacea Photos
  2. Nail Fungus and other Nail Disease
  3. Light Diseases and Disorders of Pigmentation
  4. Vascular Tumors
  5. Cellulitis Impetigo and other Bacterial Infections
  6. Scabies Lyme Disease and other Infestations and Bites
  7. Lupus and other Connective Tissue diseases




In [5]:
# Find image folders
possible_paths = [
    DERMNET_PATH,
    os.path.join(DERMNET_PATH, 'versions', '1'),
    os.path.join(DERMNET_PATH, 'train')
]

DERMNET_IMG_PATH = DERMNET_PATH
for test_path in possible_paths:
    if os.path.exists(test_path):
        subdirs = [d for d in os.listdir(test_path) if os.path.isdir(os.path.join(test_path, d))]
        if subdirs:
            first_dir = os.path.join(test_path, subdirs[0])
            files = os.listdir(first_dir)
            if any(f.lower().endswith(('.jpg', '.png')) for f in files):
                DERMNET_IMG_PATH = test_path
                break

print(f"Using path: {DERMNET_IMG_PATH}\n")

# Show which classes will be used
print("Classes available and their status:")
for cls in os.listdir(DERMNET_IMG_PATH):
    cls_path = os.path.join(DERMNET_IMG_PATH, cls)
    if os.path.isdir(cls_path):
        num_imgs = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png'))])
        if num_imgs > 0:
            status = "✅ WILL TRAIN" if cls in CLASSES_TO_KEEP else "⏭️  SKIP"
            print(f"  {status:15s} {cls}: {num_imgs} images")

print("\n")


# possible_paths = [
#     DERMNET_PATH,
#     os.path.join(DERMNET_PATH, 'versions', '1'),
#     os.path.join(DERMNET_PATH, 'train')
# ]

# DERMNET_IMG_PATH = DERMNET_PATH
# for test_path in possible_paths:
#     if os.path.exists(test_path):
#         subdirs = [d for d in os.listdir(test_path) if os.path.isdir(os.path.join(test_path, d))]
#         if subdirs:
#             first_dir = os.path.join(test_path, subdirs[0])
#             files = os.listdir(first_dir)
#             if any(f.lower().endswith(('.jpg', '.png')) for f in files):
#                 DERMNET_IMG_PATH = test_path
#                 break

# print(f"Using path: {DERMNET_IMG_PATH}")
# print("\nClasses found:")
# for cls in os.listdir(DERMNET_IMG_PATH):
#     cls_path = os.path.join(DERMNET_IMG_PATH, cls)
#     if os.path.isdir(cls_path):
#         num_imgs = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png'))])
#         if num_imgs > 0:
#             print(f"  {cls}: {num_imgs} images")

Using path: /root/.cache/kagglehub/datasets/shubhamgoel27/dermnet/versions/1/train

Classes available and their status:
  ✅ WILL TRAIN    Nail Fungus and other Nail Disease: 1040 images
  ⏭️  SKIP        Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions: 1149 images
  ⏭️  SKIP        Psoriasis pictures Lichen Planus and related diseases: 1405 images
  ⏭️  SKIP        Warts Molluscum and other Viral Infections: 1086 images
  ✅ WILL TRAIN    Lupus and other Connective Tissue diseases: 420 images
  ⏭️  SKIP        Urticaria Hives: 212 images
  ⏭️  SKIP        Hair Loss Photos Alopecia and other Hair Diseases: 239 images
  ⏭️  SKIP        Herpes HPV and other STDs Photos: 405 images
  ⏭️  SKIP        Eczema Photos: 1235 images
  ✅ WILL TRAIN    Vascular Tumors: 482 images
  ✅ WILL TRAIN    Acne and Rosacea Photos: 840 images
  ⏭️  SKIP        Vasculitis Photos: 416 images
  ✅ WILL TRAIN    Scabies Lyme Disease and other Infestations and Bites: 431 images
  ⏭️  SKIP        

In [6]:
# Create train/test split
WORK_DIR = '/content/dermnet_organized'
TRAIN_DIR = os.path.join(WORK_DIR, 'train')
TEST_DIR = os.path.join(WORK_DIR, 'test')

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Counter for statistics
total_processed = 0
skipped_classes = []

for cls_folder in tqdm(os.listdir(DERMNET_IMG_PATH), desc="Organizing images"):
    cls_path = os.path.join(DERMNET_IMG_PATH, cls_folder)

    if not os.path.isdir(cls_path):
        continue

    # ✅ ADD THIS CHECK - Only process classes in CLASSES_TO_KEEP
    if cls_folder not in CLASSES_TO_KEEP:
        skipped_classes.append(cls_folder)
        continue  # Skip this class!

    # Get all images
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png'))]

    if len(images) == 0:
        continue

    # Create class folders
    os.makedirs(os.path.join(TRAIN_DIR, cls_folder), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, cls_folder), exist_ok=True)

    # Split
    train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)

    # Copy train images
    for img in train_imgs:
        shutil.copy2(os.path.join(cls_path, img), os.path.join(TRAIN_DIR, cls_folder, img))

    # Copy test images
    for img in test_imgs:
        shutil.copy2(os.path.join(cls_path, img), os.path.join(TEST_DIR, cls_folder, img))

    total_processed += 1

print("\n" + "="*70)
print("Dataset Organization Complete!")
print("="*70)
print(f"\n✅ Processed: {total_processed} classes")
print(f"⏭️  Skipped: {len(skipped_classes)} classes")
print(f"\nSkipped classes: {', '.join(skipped_classes[:5])}..." if len(skipped_classes) > 5 else f"\nSkipped classes: {', '.join(skipped_classes)}")
print("="*70 + "\n")


##############

# Create train/test split
# WORK_DIR = '/content/dermnet_organized'
# TRAIN_DIR = os.path.join(WORK_DIR, 'train')
# TEST_DIR = os.path.join(WORK_DIR, 'test')

# os.makedirs(TRAIN_DIR, exist_ok=True)
# os.makedirs(TEST_DIR, exist_ok=True)

# for cls_folder in tqdm(os.listdir(DERMNET_IMG_PATH), desc="Organizing images"):
#     cls_path = os.path.join(DERMNET_IMG_PATH, cls_folder)

#     if not os.path.isdir(cls_path):
#         continue

#     # Get all images
#     images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png'))]

#     if len(images) == 0:
#         continue

#     # Create class folders
#     os.makedirs(os.path.join(TRAIN_DIR, cls_folder), exist_ok=True)
#     os.makedirs(os.path.join(TEST_DIR, cls_folder), exist_ok=True)

#     # Split
#     train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)

#     # Copy
#     for img in train_imgs:
#         shutil.copy2(os.path.join(cls_path, img), os.path.join(TRAIN_DIR, cls_folder, img))

#     for img in test_imgs:
#         shutil.copy2(os.path.join(cls_path, img), os.path.join(TEST_DIR, cls_folder, img))

# print("Dataset organized!")

Organizing images: 100%|██████████| 23/23 [00:00<00:00, 39.36it/s]


Dataset Organization Complete!

✅ Processed: 7 classes
⏭️  Skipped: 16 classes

Skipped classes: Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions, Psoriasis pictures Lichen Planus and related diseases, Warts Molluscum and other Viral Infections, Urticaria Hives, Hair Loss Photos Alopecia and other Hair Diseases...



In [7]:
# Setup generators
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"Classes: {class_names}")

Found 3253 images belonging to 7 classes.
Found 816 images belonging to 7 classes.
Classes: ['Acne and Rosacea Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Nail Fungus and other Nail Disease', 'Scabies Lyme Disease and other Infestations and Bites', 'Vascular Tumors']


In [8]:
# Build model
base_model = ResNet152V2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [9]:
# Callbacks
callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_SAVE_DIR, 'dermnet_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3
    )
]

In [10]:
# Train Phase 1
history1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=test_generator,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 464ms/step - accuracy: 0.3547 - loss: 2.0904
Epoch 1: val_accuracy improved from -inf to 0.53922, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/dermnet_best.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 112s 822ms/step - accuracy: 0.3553 - loss: 2.0883 - val_accuracy: 0.5392 - val_loss: 1.2924 - learning_rate: 0.0010
Epoch 2/15
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step - accuracy: 0.4523 - loss: 1.6411
Epoch 2: val_accuracy improved from 0.53922 to 0.55637, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/dermnet_best.keras
102/102 ━━━━━━━━━━━━━━━━━━━━ 46s 453ms/step - accuracy: 0.4523 - loss: 1.6409 - val_accuracy: 0.5564 - val_loss: 1.2262 - learning_rate: 0.0010
Epoch 3/15
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 377ms/step - accuracy: 0.4924 - loss: 1.4675
Epoch 3: val_accuracy improved from 0.55637 to 0.57475, saving model to /content/drive/MyDrive/CMPE295 Project/ML/Ensemble/Models/dermnet_best.keras
102

In [11]:
# Train Phase 2
base_model.trainable = True
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=13,
    validation_data=test_generator,
    callbacks=callbacks
)

Epoch 1/13
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5260 - loss: 1.3097
Epoch 1: val_accuracy did not improve from 0.60539
102/102 ━━━━━━━━━━━━━━━━━━━━ 268s 1s/step - accuracy: 0.5262 - loss: 1.3093 - val_accuracy: 0.5135 - val_loss: 1.3958 - learning_rate: 1.0000e-04
Epoch 2/13
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 706ms/step - accuracy: 0.5777 - loss: 1.1628
Epoch 2: val_accuracy did not improve from 0.60539
102/102 ━━━━━━━━━━━━━━━━━━━━ 78s 758ms/step - accuracy: 0.5778 - loss: 1.1627 - val_accuracy: 0.6017 - val_loss: 1.1332 - learning_rate: 1.0000e-04
Epoch 3/13
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 705ms/step - accuracy: 0.6085 - loss: 1.0652
Epoch 3: val_accuracy did not improve from 0.60539
102/102 ━━━━━━━━━━━━━━━━━━━━ 77s 753ms/step - accuracy: 0.6086 - loss: 1.0654 - val_accuracy: 0.5478 - val_loss: 1.2902 - learning_rate: 1.0000e-04
Epoch 4/13
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 704ms/step - accuracy: 0.6275 - loss: 1.0049
Epoch 4: val_accuracy improved from 0.60539 to 0.61152, 

In [12]:
# Evaluate and save
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")

model.save(os.path.join(MODEL_SAVE_DIR, 'dermnet_final.keras'))

with open(os.path.join(MODEL_SAVE_DIR, 'dermnet_classes.json'), 'w') as f:
    json.dump(class_names, f)

model_info = {
    'dataset': 'DermNet',
    'num_classes': num_classes,
    'classes': class_names,
    'test_accuracy': float(test_accuracy),
    'image_type': 'clinical',
    'input_size': IMG_SIZE
}

with open(os.path.join(MODEL_SAVE_DIR, 'dermnet_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("\n✅ Model saved to Google Drive!")

26/26 ━━━━━━━━━━━━━━━━━━━━ 5s 185ms/step - accuracy: 0.7051 - loss: 0.9053

Test Accuracy: 71.32%

✅ Model saved to Google Drive!
